# 60-EPOCH BUDGET VERSION
Total training = **60 epochs**: `default` and `ot_nwd` x seeds 0, 1 x 15 epochs. Gives mean +- std over 2 seeds for the key comparison. Same epochs for both modes. Outputs: `results_budget60.csv`, `runs_budget60/`. Resumable: if the session dies, rerun and finished runs are skipped. Save Version at the end, because `/kaggle/working` is wiped between sessions.

---
# Final experiments: entropic-OT label assignment with a Gaussian-Wasserstein cost (VisDrone-DET, sequence-level 70/15/15 split)
Settings: **GPU on, Internet on**. Single GPU only (`device=0`), because the assigner is patched into this process.
Finished runs are skipped (weights on disk + `results.csv`), so the notebook can be re-run after a crash or in several sessions.

In [6]:
!pip -q install "ultralytics==8.3.0" --no-deps
!pip -q install py-cpuinfo ultralytics-thop
import pycocotools, ultralytics; print("ultralytics", ultralytics.__version__)

ultralytics 8.3.0


In [7]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'   # must be set before the first CUDA allocation
import sys, gc, shutil, random, json, time
from pathlib import Path
import numpy as np, pandas as pd, torch
from PIL import Image

# ======================= experiment settings =======================
EPOCHS = 15            # per run. BUDGET: 2 modes x 2 seeds x 15 epochs = 60 epochs in total
IMGSZ  = 960           # 640 shrinks VisDrone objects too much (Ultralytics still accumulates to nominal batch 64)
BATCH  = 4             # 8 runs out of memory on a T4 at 960 px: the stock assigner's tensor grows with batch x gt-count x anchors
SEEDS  = [0, 1]
REUSE  = True          # skip runs that already have best.pt AND a row in results.csv
RUN_NOW = None         # None = all 4 runs (60 epochs). To split over sessions use e.g. [('default', 0), ('ot_nwd', 0)] then [('default', 1), ('ot_nwd', 1)]
#RUN_NOW = [('default', 0), ('ot_nwd', 0)]   # ONLY these (mode, seed) runs are executed this session; set None to run everything
PREV_OUTPUT = None     # results.csv from an earlier session's saved output, e.g. '/kaggle/input/<your-notebook>/results.csv'
DO_SWEEP = False       # sensitivity runs (seed 0 only); switch on after the main table is done
SPLIT_SEED = 0

# name -> assigner config (None = stock Ultralytics TaskAlignedAssigner)
MODE_CFG = {
    'default':        None,                                              # baseline (TAL)
    'ot_nwd':         dict(cost='nwd', target='nwd'),                    # proposed
    'ot_nwd_adapt':   dict(cost='nwd', target='nwd', nwd_adapt=True),    # proposed, scale-adaptive normaliser
    'ot_nwd_iou_tgt': dict(cost='nwd', target='iou'),                    # ablation: only the soft target is IoU
    'ot_iou':         dict(cost='iou', target='iou'),                    # ablation: OT with an IoU cost
}
MODES = ['default', 'ot_nwd']   # BUDGET: only baseline vs proposed. Add 'ot_nwd_adapt', 'ot_iou', 'ot_nwd_iou_tgt' only if quota allows (same EPOCHS)
SWEEP = {                                                                # sensitivity of the proposed method
    'ot_nwd_c6':    dict(cost='nwd', target='nwd', nwd_c=6.0),
    'ot_nwd_c25':   dict(cost='nwd', target='nwd', nwd_c=25.0),
    'ot_nwd_eps05': dict(cost='nwd', target='nwd', ot_eps=0.05),
    'ot_nwd_eps5':  dict(cost='nwd', target='nwd', ot_eps=0.5),
}
# ===================================================================
WORK = Path('/kaggle/working'); DATA = WORK/'visdrone'; RUNS = str(WORK/'runs_budget60'); RESULTS = WORK/'results_budget60.csv'
sys.path.insert(0, str(WORK))

# Ultralytics' third-party logging callbacks (wandb, ray, ...) crash on Kaggle's package versions; switch them off
os.environ['WANDB_DISABLED'] = 'true'; os.environ['WANDB_MODE'] = 'disabled'
from ultralytics import settings
settings.update({k: False for k in ('wandb', 'raytune', 'clearml', 'comet', 'dvc', 'mlflow', 'neptune')})

def free_gpu():
    gc.collect(); torch.cuda.empty_cache()

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable a GPU in Session options')

GPU: Tesla T4


In [8]:
# ---------------- convert VisDrone-train to YOLO format, then split 70/15/15 by video sequence ----------------
def find(name):
    return next(Path('/kaggle/input').rglob(name))

def convert(src, dst):
    src, dst = Path(src), Path(dst)
    (dst/'images').mkdir(parents=True, exist_ok=True); (dst/'labels').mkdir(parents=True, exist_ok=True)
    n = 0
    for img in sorted((src/'images').glob('*.jpg')):
        link = dst/'images'/img.name
        if not link.exists():
            try: os.symlink(img, link)
            except OSError: shutil.copy(img, link)
        W, H = Image.open(img).size
        lines = []
        for row in (src/'annotations'/f'{img.stem}.txt').read_text().strip().splitlines():
            x, y, w, h, score, cat = map(int, row.split(',')[:6])
            if score == 0 or cat in (0, 11):          # ignored regions / "others"
                continue
            lines.append(f"{cat-1} {(x+w/2)/W:.6f} {(y+h/2)/H:.6f} {w/W:.6f} {h/H:.6f}")
        (dst/'labels'/f'{img.stem}.txt').write_text("\n".join(lines)); n += 1
    print('converted', n, 'images')

if not (DATA/'test'/'images').exists():            # build the split once; later runs reuse it
    shutil.rmtree(DATA, ignore_errors=True)
    convert(find('VisDrone2019-DET-train'), DATA/'all')
    # frames of one clip share the 7-digit prefix -> split by clip so near-duplicate frames never cross splits
    seqs = sorted({p.stem.split('_')[0] for p in (DATA/'all'/'images').glob('*.jpg')})
    random.Random(SPLIT_SEED).shuffle(seqs)
    n = len(seqs); a, b = int(0.70 * n), int(0.85 * n)
    split_of = {q: 'train' for q in seqs[:a]} | {q: 'val' for q in seqs[a:b]} | {q: 'test' for q in seqs[b:]}
    for sp in ('train', 'val', 'test'):
        (DATA/sp/'images').mkdir(parents=True, exist_ok=True); (DATA/sp/'labels').mkdir(parents=True, exist_ok=True)
    for img in list((DATA/'all'/'images').glob('*.jpg')):
        sp = split_of[img.stem.split('_')[0]]
        os.replace(img, DATA/sp/'images'/img.name)
        os.replace(DATA/'all'/'labels'/f'{img.stem}.txt', DATA/sp/'labels'/f'{img.stem}.txt')
    shutil.rmtree(DATA/'all')
for sp in ('train', 'val', 'test'):
    print(sp, len(list((DATA/sp/'images').glob('*.jpg'))), 'images')

NAMES = ['pedestrian','people','bicycle','car','van','truck','tricycle','awning-tricycle','bus','motor']
YAML = str(WORK/'visdrone.yaml')
Path(YAML).write_text(f"path: {DATA}\ntrain: train/images\nval: val/images\ntest: test/images\nnames:\n" +
                      "".join(f"  {i}: {nm}\n" for i, nm in enumerate(NAMES)))

train 4707 images
val 474 images
test 1290 images


217

## Assigner module

In [9]:
%%writefile /kaggle/working/ot_assigner.py
"""
Entropic-OT label assignment with a Gaussian-Wasserstein (NWD) cost for Ultralytics YOLOv8.

Math (matches the paper draft):
  - box -> phi(b) = (cx, cy, w/2, h/2). For axis-aligned Gaussians, W2(N_i, N_j) = ||phi_i - phi_j||_2.
  - loc cost  c_loc = 1 - exp(-W2 / C)      (Lipschitz in box coordinates with constant 1/C, any object size)
  - cls cost  c_cls = -logit(p_ic)          (= BCE_gt - BCE_bg; row-constant terms cancel in Sinkhorn)
  - C_ij = cls_w * c_cls + loc_w * c_loc ;  background column has cost 0
  - plan  pi = argmin <C,pi> - eps*H(pi) over U(a,b)   (log-domain Sinkhorn)
  - gt j takes its round(k_j) anchors with the largest plan mass (>=1 each); ties between gts go to the larger mass
  - cost='iou' switches c_loc to 1 - IoU (ablation); target='iou' switches only the soft cls target to IoU
  - nwd_adapt=True uses C_j = max(nwd_c, rho*sqrt(w_j h_j)): scale-aware, Lipschitz constant still <= 1/nwd_c
"""
import torch
from torchvision.ops import box_iou
from ultralytics.utils.tal import TaskAlignedAssigner

OT_CFG = dict(
    ot_eps=0.1,          # entropic regularisation epsilon
    sink_iters=50,       # Sinkhorn iterations
    nwd_c=12.8,          # NWD normalising constant (tune; dataset dependent)
    cls_w=1.0,
    loc_w=3.0,
    center_radius=2.5,   # candidate anchors: centre within R strides of gt centre
    min_cand=9,          # every gt always gets at least this many nearest anchors
    max_k=10,            # cap on dynamic k (demand of each gt)
    strides=(8, 16, 32),
    cost="nwd",          # similarity used in the transport cost: "nwd" or "iou"
    target="nwd",        # similarity used for the soft classification target: "nwd" or "iou"
    nwd_adapt=False,     # scale-adaptive normaliser C_j = max(nwd_c, nwd_rho * sqrt(w_j h_j)); Lipschitz const <= 1/nwd_c
    nwd_rho=0.25,
    big=1e4,             # cost of non-candidate pairs
)
_DEFAULT_CFG = dict(OT_CFG)


def phi(b):
    """xyxy -> (cx, cy, w/2, h/2)"""
    cx = (b[..., 0] + b[..., 2]) / 2
    cy = (b[..., 1] + b[..., 3]) / 2
    hw = (b[..., 2] - b[..., 0]) / 2
    hh = (b[..., 3] - b[..., 1]) / 2
    return torch.stack([cx, cy, hw, hh], -1)


def w2_matrix(pred_xyxy, gt_xyxy):
    """(M,4),(G,4) -> (M,G) Wasserstein-2 distance between Gaussian boxes."""
    return torch.cdist(phi(pred_xyxy), phi(gt_xyxy))


def sinkhorn_log(C, a, b, eps, iters):
    """Log-domain Sinkhorn. C:(M,N), a:(M,), b:(N,) positive, each summing to 1. Returns plan (M,N)."""
    logK = -C / eps
    la, lb = a.log(), b.log()
    f = torch.zeros_like(a)
    g = torch.zeros_like(b)
    for _ in range(iters):
        f = la - torch.logsumexp(logK + g[None, :], dim=1)
        g = lb - torch.logsumexp(logK + f[:, None], dim=0)
    return torch.exp(logK + f[:, None] + g[None, :])


def infer_stride(anc, strides=(8, 16, 32)):
    """Anchor centres are (k+0.5)*s, so the stride is recoverable from the x coordinate."""
    x = anc[:, 0]
    out = torch.full_like(x, float(strides[-1]))
    for s in reversed(strides):
        frac = (x / s) % 1.0
        out = torch.where((frac - 0.5).abs() < 1e-3, torch.full_like(x, float(s)), out)
    return out


class OTAssigner(TaskAlignedAssigner):
    """Drop-in replacement for TaskAlignedAssigner inside v8DetectionLoss."""

    def __init__(self, topk=10, num_classes=80, alpha=0.5, beta=6.0, eps=1e-9, **kw):
        super().__init__(topk=topk, num_classes=num_classes, alpha=alpha, beta=beta, eps=eps)
        self.cfg = dict(OT_CFG)

    # ---- one image -------------------------------------------------------------------
    def _assign_one(self, sc, pb, anc, strd, gb, gl):
        cfg = self.cfg
        G, A = gb.shape[0], pb.shape[0]
        gc = torch.stack([(gb[:, 0] + gb[:, 2]) / 2, (gb[:, 1] + gb[:, 3]) / 2], -1)
        nd = torch.cdist(gc, anc) / strd[None, :]                       # (G,A) distance in strides
        cand = nd <= cfg["center_radius"]
        top = nd.topk(min(cfg["min_cand"], A), dim=1, largest=False).indices
        cand.scatter_(1, top, True)
        cidx = cand.any(0).nonzero().squeeze(-1)                         # candidate anchors
        M = cidx.numel()
        cand = cand[:, cidx].T                                           # (M,G)
        pbc = pb[cidx]
        p = sc[cidx].clamp(1e-4, 1 - 1e-4)

        W2 = w2_matrix(pbc, gb)
        if cfg["nwd_adapt"]:
            area = ((gb[:, 2] - gb[:, 0]) * (gb[:, 3] - gb[:, 1])).clamp(min=1.0)
            Cj = (cfg["nwd_rho"] * area.sqrt()).clamp(min=cfg["nwd_c"])          # (G,)
        else:
            Cj = torch.full((G,), float(cfg["nwd_c"]), device=gb.device)
        nwd = torch.exp(-W2 / Cj[None, :])
        iou_m = box_iou(pbc, gb)
        sim = iou_m if cfg["cost"] == "iou" else nwd                              # drives cost and dynamic k
        tsim = iou_m if cfg["target"] == "iou" else nwd                           # drives the soft cls target
        pc = p[:, gl]                                                    # (M,G) score of each gt's class
        c_cls = -(pc.log() - (1 - pc).log())
        C = cfg["cls_w"] * c_cls + cfg["loc_w"] * (1 - sim)
        C = torch.where(cand, C, torch.full_like(C, cfg["big"]))

        # dynamic k (OTA / SimOTA style, on the chosen similarity)
        simc = torch.where(cand, sim, torch.zeros_like(sim))
        k = simc.topk(min(cfg["max_k"], M), dim=0).values.sum(0).clamp(1.0, cfg["max_k"])
        if k.sum() > 0.9 * M:
            k = k * (0.9 * M / k.sum())

        a = torch.full((M,), 1.0 / M, device=C.device)
        b = torch.cat([(M - k.sum())[None], k]) / M                      # [background, gt_1..gt_G]
        C_full = torch.cat([torch.zeros(M, 1, device=C.device), C], 1)
        plan = sinkhorn_log(C_full, a, b, cfg["ot_eps"], cfg["sink_iters"])

        # positives: for every gt take its round(k_j) anchors with the largest plan mass (>=1 per gt);
        # an anchor claimed by several gts keeps the gt with the largest mass.
        col = plan[:, 1:]                                                # (M,G)
        kk = k.round().clamp(min=1).long()
        order = col.argsort(0, descending=True)                          # (M,G)
        rank_ok = torch.arange(M, device=C.device)[:, None] < kk[None, :]
        P = torch.zeros_like(col, dtype=torch.bool).scatter_(0, order, rank_ok)
        j = torch.where(P, col, torch.full_like(col, -1.0)).argmax(1)
        pos = P.any(1)
        return cidx[pos], j[pos], tsim[pos, j[pos]]

    # ---- batch -----------------------------------------------------------------------
    @torch.no_grad()
    def forward(self, pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt):
        self.bs = pd_scores.shape[0]
        self.n_max_boxes = gt_bboxes.shape[1]
        device = gt_bboxes.device
        if self.n_max_boxes == 0:
            return (
                torch.full_like(pd_scores[..., 0], self.bg_idx),
                torch.zeros_like(pd_bboxes),
                torch.zeros_like(pd_scores),
                torch.zeros_like(pd_scores[..., 0]).bool(),
                torch.zeros_like(pd_scores[..., 0]).long(),
            )
        A = pd_scores.shape[1]
        pd_scores, pd_bboxes, anc_points = pd_scores.float(), pd_bboxes.float(), anc_points.float()
        gt_bboxes = gt_bboxes.float()
        strd = infer_stride(anc_points, self.cfg["strides"])

        mask_pos = torch.zeros(self.bs, self.n_max_boxes, A, device=device)
        overlaps = torch.zeros_like(mask_pos)
        align = torch.zeros_like(mask_pos)

        for bi in range(self.bs):
            gidx = mask_gt[bi, :, 0].bool().nonzero().squeeze(-1)
            if gidx.numel() == 0:
                continue
            gl = gt_labels[bi, gidx, 0].long()
            ai, gj, sim = self._assign_one(pd_scores[bi], pd_bboxes[bi], anc_points, strd, gt_bboxes[bi, gidx], gl)
            gi = gidx[gj]
            mask_pos[bi, gi, ai] = 1.0
            overlaps[bi, gi, ai] = sim
            score = pd_scores[bi, ai, gl[gj]]
            align[bi, gi, ai] = score.pow(self.alpha) * sim.pow(self.beta)

        fg_mask = mask_pos.sum(-2)                                       # (b,A)
        target_gt_idx = mask_pos.argmax(-2)                              # (b,A)
        target_labels, target_bboxes, target_scores = self.get_targets(gt_labels, gt_bboxes, target_gt_idx, fg_mask)

        pos_align = align.amax(-1, keepdim=True)
        pos_ov = overlaps.amax(-1, keepdim=True)
        norm = (align * pos_ov / (pos_align + self.eps)).amax(-2).unsqueeze(-1)
        target_scores = target_scores * norm
        return target_labels, target_bboxes, target_scores, fg_mask.bool(), target_gt_idx


# ---- switching between default and OT assigner --------------------------------------
def use_ot(**overrides):
    import ultralytics.utils.loss as L
    OT_CFG.clear(); OT_CFG.update(_DEFAULT_CFG); OT_CFG.update(overrides)       # no leakage between runs
    L.TaskAlignedAssigner = OTAssigner


def use_default():
    import ultralytics.utils.loss as L
    L.TaskAlignedAssigner = TaskAlignedAssigner

Overwriting /kaggle/working/ot_assigner.py


In [10]:
# ---------------- smoke test: every assigner variant on synthetic tensors (run before training) ----------------
import importlib, ot_assigner as OT
importlib.reload(OT)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
b, nmax, nc, img = 2, 4, 10, 640
anc = []
for s in (8, 16, 32):
    n = img // s; xs = (torch.arange(n) + 0.5) * s
    gy, gx = torch.meshgrid(xs, xs, indexing='ij'); anc.append(torch.stack([gx.flatten(), gy.flatten()], -1))
anc = torch.cat(anc).to(dev); A = anc.shape[0]
c = torch.rand(b, nmax, 2, device=dev) * 500 + 60; wh = torch.tensor([6., 12., 30., 90.], device=dev)[None, :, None].expand(b, nmax, 2)
gt_bboxes = torch.cat([c - wh / 2, c + wh / 2], -1)
gt_labels = torch.randint(0, nc, (b, nmax, 1), device=dev).float()
mask_gt = torch.ones(b, nmax, 1, device=dev); mask_gt[1, 3] = 0
pd_scores = torch.rand(b, A, nc, device=dev) * 0.05
pd_bboxes = torch.cat([anc - 8, anc + 8], -1)[None].expand(b, -1, -1).contiguous()
for name, cfg in MODE_CFG.items():
    if cfg is None: continue
    OT.use_ot(**cfg)
    tl, tb, ts, fg, gi = OT.OTAssigner(topk=10, num_classes=nc)(pd_scores, pd_bboxes, anc, gt_labels, gt_bboxes, mask_gt)
    for bi in range(b):
        for g in range(nmax):
            if mask_gt[bi, g, 0] > 0:
                assert (gi[bi][fg[bi]] == g).any(), f'{name}: gt {g} of image {bi} got no positive anchor'
    assert torch.isfinite(ts).all() and ts.sum() > 0
    print(f'{name:16s} ok | positives per image {fg.sum(1).tolist()} | target_scores sum {float(ts.sum()):.2f}')
OT.use_default(); print('smoke test passed')

ot_nwd           ok | positives per image [14, 13] | target_scores sum 6.84
ot_nwd_adapt     ok | positives per image [14, 13] | target_scores sum 6.93
ot_nwd_iou_tgt   ok | positives per image [14, 13] | target_scores sum 3.26
ot_iou           ok | positives per image [6, 6] | target_scores sum 2.55
smoke test passed


## Stability experiment (Theorem 2 / Proposition 3): numpy, no training needed

In [11]:
%%writefile /kaggle/working/stability_exp.py
import numpy as np
from scipy.special import logsumexp


def phi(b):
    return np.stack([(b[..., 0] + b[..., 2]) / 2, (b[..., 1] + b[..., 3]) / 2,
                     (b[..., 2] - b[..., 0]) / 2, (b[..., 3] - b[..., 1]) / 2], -1)


def w2(pred, gt):
    d = phi(pred)[:, None, :] - phi(gt)[None, :, :]
    return np.sqrt((d ** 2).sum(-1))


def iou(pred, gt):
    lt = np.maximum(pred[:, None, :2], gt[None, :, :2])
    rb = np.minimum(pred[:, None, 2:], gt[None, :, 2:])
    inter = np.clip(rb - lt, 0, None).prod(-1)
    ap = (pred[:, 2:] - pred[:, :2]).prod(-1)[:, None]
    ag = (gt[:, 2:] - gt[:, :2]).prod(-1)[None, :]
    return inter / (ap + ag - inter + 1e-12)


def sinkhorn_log(C, a, b, eps, iters=200):
    logK = -C / eps
    la, lb = np.log(a), np.log(b)
    f, g = np.zeros_like(a), np.zeros_like(b)
    for _ in range(iters):
        f = la - logsumexp(logK + g[None, :], axis=1)
        g = lb - logsumexp(logK + f[:, None], axis=0)
    return np.exp(logK + f[:, None] + g[None, :])


def anchors(img=640, strides=(8, 16, 32)):
    pts = []
    for s in strides:
        n = img // s
        xs = (np.arange(n) + 0.5) * s
        gx, gy = np.meshgrid(xs, xs)
        pts.append(np.stack([gx.ravel(), gy.ravel()], -1))
    return np.concatenate(pts)


def make_scene(rng, s, G=5, img=640, anc=None):
    c = rng.uniform(64, img - 64, (G, 2))
    gt = np.concatenate([c - s / 2, c + s / 2], 1)
    # one prediction per anchor: centred near the anchor, size ~ s
    sz = s * rng.uniform(0.8, 1.2, (len(anc), 1))
    ctr = anc + rng.normal(0, s / 4, anc.shape)
    pred = np.concatenate([ctr - sz / 2, ctr + sz / 2], 1)
    return pred, gt


def plan_for(pred, gt, kind, eps, nwd_c, k=3.0):
    sim = np.exp(-w2(pred, gt) / nwd_c) if kind == "nwd" else iou(pred, gt)
    C = np.concatenate([np.zeros((len(pred), 1)), 1.0 - sim], 1)   # bg column cost 0, constant cls cost dropped
    M, G = len(pred), gt.shape[0]
    a = np.full(M, 1.0 / M)
    b = np.concatenate([[M - k * G], np.full(G, k)]) / M
    return sinkhorn_log(C, a, b, eps), sim


def perturb(pred, delta, rng):
    ang = rng.uniform(0, 2 * np.pi, len(pred))
    d = delta * np.stack([np.cos(ang), np.sin(ang)], 1)
    return pred + np.concatenate([d, d], 1)          # shift each box by exactly delta pixels


def run(sizes=(4, 8, 16, 32, 64), delta=1.0, trials=20, eps=0.1, nwd_c=12.8, seed=0):
    """Per object size s:
       l1_*   : ||pi - pi'||_1 / delta after a delta-pixel jitter of every predicted box (Theorem 2; bound = 1/(C*eps) for NWD)
       dist_* : mean centre distance (in object sizes) of the top-3 anchors the plan gives each gt: is the supervision located on the object?
       flip   : hard IoU>=0.5 positive set, fraction of positives that flip under the same jitter."""
    rng = np.random.default_rng(seed)
    anc = anchors()
    rows = []
    for s in sizes:
        acc = {k: [] for k in ("l1_nwd", "l1_iou", "npos_nwd", "npos_iou", "flip")}
        for _ in range(trials):
            pred, gt = make_scene(rng, s, anc=anc)
            pred2 = perturb(pred, delta, rng)
            for kind in ("nwd", "iou"):
                p1, _ = plan_for(pred, gt, kind, eps, nwd_c)
                p2, _ = plan_for(pred2, gt, kind, eps, nwd_c)
                acc["l1_" + kind].append(np.abs(p1 - p2).sum() / delta)
                col = p1[:, 1:]
                top = np.argsort(-col, axis=0)[:3]                     # top-3 anchors per gt (k=3)
                gc = (gt[:, :2] + gt[:, 2:]) / 2
                pc = (pred[:, :2] + pred[:, 2:]) / 2
                dist = np.linalg.norm(pc[top] - gc[None], axis=2) / s  # (3,G) centre distance in object sizes
                acc["npos_" + kind].append(dist.mean())
            m1, m2 = iou(pred, gt).max(1) >= 0.5, iou(pred2, gt).max(1) >= 0.5
            u = m1 | m2
            acc["flip"].append((m1[u] != m2[u]).mean() if u.any() else np.nan)
        rows.append((s,) + tuple(np.nanmean(acc[k]) for k in ("l1_nwd", "l1_iou", "npos_nwd", "npos_iou", "flip")))
    return rows, 1.0 / (nwd_c * eps)


if __name__ == "__main__":
    # 1) W2 identity check against the closed-form Gaussian Wasserstein-2
    from scipy.linalg import sqrtm
    rng = np.random.default_rng(1)
    err = 0
    for _ in range(200):
        b1 = np.sort(rng.uniform(0, 100, 4)).reshape(2, 2).T.ravel()[[0, 2, 1, 3]]
        b1 = np.array([rng.uniform(0, 50), rng.uniform(0, 50), 0, 0]); b1[2:] = b1[:2] + rng.uniform(2, 40, 2)
        b2 = np.array([rng.uniform(0, 50), rng.uniform(0, 50), 0, 0]); b2[2:] = b2[:2] + rng.uniform(2, 40, 2)
        mu1, mu2 = (b1[:2] + b1[2:]) / 2, (b2[:2] + b2[2:]) / 2
        S1, S2 = np.diag(((b1[2:] - b1[:2]) / 2) ** 2), np.diag(((b2[2:] - b2[:2]) / 2) ** 2)
        S2h = sqrtm(S2)
        cross = sqrtm(S2h @ S1 @ S2h)
        closed = np.sqrt(((mu1 - mu2) ** 2).sum() + np.trace(S1 + S2 - 2 * cross).real)
        err = max(err, abs(closed - w2(b1[None], b2[None])[0, 0]))
    print(f"max |closed-form W2 - ||phi diff||| over 200 pairs: {err:.2e}")

    # 2) Proposition 3: IoU slope for shifted s x s boxes
    for s in (4, 8, 16, 64):
        d = 1e-3
        a = np.array([[0, 0, s, s]], float); b = np.array([[d, 0, s + d, s]], float)
        print(f"s={s}: |dIoU/dx| ~ {abs(1 - iou(a, b)[0, 0]) / d:.4f}  (2/s = {2 / s:.4f})")

    # 3) Stability + supervision vs object size
    rows, bound = run()
    print(f"\nTheorem-2 bound for NWD cost: ||pi-pi'||_1 / delta <= 1/(C*eps) = {bound:.3f}")
    print(f"{'size':>5} {'L1 NWD':>9} {'L1 IoU':>9} {'dist/s NWD':>11} {'dist/s IoU':>11} {'hard flip':>10}")
    for r in rows:
        print(f"{r[0]:>5} " + " ".join(f"{v:>10.4f}" for v in r[1:]))

Writing /kaggle/working/stability_exp.py


In [12]:
import stability_exp as SE
importlib.reload(SE)
rows_s, bound = SE.run(trials=50)
stab = pd.DataFrame(rows_s, columns=['object_size_px', 'L1_plan_change_NWD', 'L1_plan_change_IoU',
                                     'dist_to_gt_in_sizes_NWD', 'dist_to_gt_in_sizes_IoU', 'hard_IoU0.5_flip_rate'])
stab.to_csv(WORK/'stability_results.csv', index=False)
print(f'Theorem-2 bound for the NWD cost: L1/delta <= 1/(C*eps) = {bound:.3f}')
print(stab.round(4).to_string(index=False))

Theorem-2 bound for the NWD cost: L1/delta <= 1/(C*eps) = 0.781
 object_size_px  L1_plan_change_NWD  L1_plan_change_IoU  dist_to_gt_in_sizes_NWD  dist_to_gt_in_sizes_IoU  hard_IoU0.5_flip_rate
              4              0.0003              0.0000                   1.1786                  59.4597                 0.8864
              8              0.0003              0.0001                   0.5872                   0.6955                 0.5356
             16              0.0003              0.0002                   0.3091                   0.3113                 0.2767
             32              0.0002              0.0002                   0.1565                   0.1574                 0.1367
             64              0.0002              0.0003                   0.0868                   0.0878                 0.0688


## Training and evaluation (COCO-style AP with tiny-object size ranges, on the held-out test split)

In [13]:
from ultralytics import YOLO
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

def train_run(name, cfg, seed):
    import ot_assigner as OT
    OT.use_default() if cfg is None else OT.use_ot(**cfg)
    seed_all(seed)
    model = YOLO('yolov8n.pt')
    model.train(data=YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=seed, device=0, workers=2,
                project=RUNS, name=name, exist_ok=True, plots=False, deterministic=False, verbose=False)
    best = str(model.trainer.best)
    del model; free_gpu()
    return best

AREAS = [[0, 1e10], [0, 8**2], [8**2, 16**2], [16**2, 32**2], [32**2, 1e10]]
LBLS  = ['AP', 'AP_vtiny(<8)', 'AP_tiny(8-16)', 'AP_small(16-32)', 'AP_rest(>32)']

def _ap(E, a, t=None):
    pr = E.eval['precision'][:, :, :, a, -1]
    if t is not None: pr = pr[t:t+1]
    v = pr[pr > -1]
    return float(v.mean()) * 100 if v.size else float('nan')

def coco_eval(weights, split):
    imgs = sorted((split/'images').glob('*.jpg'))
    gt = {'images': [], 'annotations': [], 'categories': [{'id': i} for i in range(10)]}
    aid = 1
    for iid, p in enumerate(imgs, 1):
        W, H = Image.open(p).size
        gt['images'].append({'id': iid, 'width': W, 'height': H})
        lab = split/'labels'/f'{p.stem}.txt'
        for row in lab.read_text().split('\n') if lab.exists() else []:
            if not row.strip(): continue
            c, x, y, w, h = row.split(); c = int(c); x, y, w, h = float(x)*W, float(y)*H, float(w)*W, float(h)*H
            gt['annotations'].append({'id': aid, 'image_id': iid, 'category_id': c, 'bbox': [x-w/2, y-h/2, w, h],
                                      'area': w*h, 'iscrowd': 0}); aid += 1
    free_gpu(); model = YOLO(weights); dets = []
    name2id = {p.name: i for i, p in enumerate(imgs, 1)}
    # NOTE: pass the image FOLDER, not a list of paths. Ultralytics treats a list as ONE batch holding every image (-> OOM).
    for r in model.predict(str(split/'images'), stream=True, imgsz=IMGSZ, conf=0.001,
                           iou=0.6, max_det=300, batch=8, half=True, verbose=False, device=0):
        iid = name2id[Path(r.path).name]
        xyxy = r.boxes.xyxy.cpu().numpy(); cf = r.boxes.conf.cpu().numpy(); cl = r.boxes.cls.cpu().numpy()
        for bb, s, c in zip(xyxy, cf, cl):
            dets.append({'image_id': iid, 'category_id': int(c), 'score': float(s),
                         'bbox': [float(bb[0]), float(bb[1]), float(bb[2]-bb[0]), float(bb[3]-bb[1])]})
    del model; free_gpu()
    G = COCO(); G.dataset = gt; G.createIndex(); D = G.loadRes(dets)
    E = COCOeval(G, D, 'bbox'); E.params.areaRng = AREAS; E.params.areaRngLbl = ['all','vtiny','tiny','small','rest']
    E.params.maxDets = [1, 10, 300]; E.evaluate(); E.accumulate()
    out = {l: _ap(E, i) for i, l in enumerate(LBLS)}; out['AP50'] = _ap(E, 0, 0)
    return out

In [14]:
# ---------------- main experiment (resumable) ----------------
todo = [(m, MODE_CFG[m], s) for m in MODES for s in SEEDS]
if DO_SWEEP:
    todo += [(k, v, SEEDS[0]) for k, v in SWEEP.items()]

if PREV_OUTPUT and Path(PREV_OUTPUT).exists() and not RESULTS.exists():
    shutil.copy(PREV_OUTPUT, RESULTS)                       # carry earlier sessions' results forward
if RUN_NOW:
    lookup = {**MODE_CFG, **SWEEP}
    todo = [(m, lookup[m], sd) for m, sd in RUN_NOW]
print('this session will run:', [(m, sd) for m, _, sd in todo])

rows = pd.read_csv(RESULTS).to_dict('records') if (REUSE and RESULTS.exists()) else []
done = {(r['mode'], r['seed']) for r in rows}

for mode, cfg, seed in todo:
    name = f'{mode}_s{seed}'
    if REUSE and (mode, seed) in done:
        print('skip', name); continue
    t0 = time.time()
    w = Path(RUNS)/name/'weights'/'best.pt'
    if not (REUSE and w.exists()):
        w = train_run(name, cfg, seed)
    t_train = time.time() - t0
    r = coco_eval(w, DATA/'test'); r.update(mode=mode, seed=seed, train_min=round(t_train/60, 1), epochs=EPOCHS, imgsz=IMGSZ)
    rows.append(r)
    pd.DataFrame(rows).to_csv(RESULTS, index=False)            # saved after every run
    print(name, {k: round(v, 2) for k, v in r.items() if isinstance(v, float)})

this session will run: [('default', 0), ('default', 1), ('ot_nwd', 0), ('ot_nwd', 1)]


100%|██████████| 6.25M/6.25M [00:00<00:00, 25.5MB/s]


New https://pypi.org/project/ultralytics/8.4.177 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.0 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/kaggle/working/visdrone.yaml, epochs=15, time=None, patience=100, batch=4, imgsz=960, save=True, save_period=-1, cache=False, device=0, workers=2, project=/kaggle/working/runs_budget60, name=default_s0, exist_ok=True, pretrained=True, optimizer=auto, verbose=False, seed=0, deterministic=False, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=False, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embe

100%|██████████| 755k/755k [00:00<00:00, 5.29MB/s]


Overriding model.yaml nc=80 with nc=10

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytic

train: Scanning /kaggle/working/visdrone/train/labels... 4707 images, 0 backgrounds, 0 corrupt: 100%|██████████| 4707/4707 [00:04<00:00, 1172.29it/s]

... (192 outputs left out)

                   all        474      27919      0.356      0.311       0.27      0.147



15 epochs completed in 0.757 hours.
Optimizer stripped from /kaggle/working/runs_budget60/ot_nwd_s1/weights/last.pt, 5.7MB
Optimizer stripped from /kaggle/working/runs_budget60/ot_nwd_s1/weights/best.pt, 5.7MB

Validating /kaggle/working/runs_budget60/ot_nwd_s1/weights/best.pt...
Ultralytics 8.3.0 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 186 layers, 2,686,318 parameters, 0 gradients, 6.8 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 60/60 [00:04<00:00, 13.62it/s]


                   all        474      27919      0.357      0.311       0.27      0.147
Speed: 0.3ms preprocess, 2.7ms inference, 0.0ms loss, 2.9ms postprocess per image
creating index...
index created!
Loading and preparing results...
DONE (t=1.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=114.38s).
Accumulating evaluation results...
DONE (t=4.15s).
ot_nwd_s1 {'AP': 15.42, 'AP_vtiny(<8)': 0.4, 'AP_tiny(8-16)': 3.07, 'AP_small(16-32)': 9.49, 'AP_rest(>32)': 24.73, 'AP50': 27.58, 'train_min': 45.9}


In [15]:
# ---------------- results table for the paper ----------------
df = pd.read_csv(RESULTS)
cols = ['AP', 'AP50', 'AP_vtiny(<8)', 'AP_tiny(8-16)', 'AP_small(16-32)', 'AP_rest(>32)']
g = df.groupby('mode')[cols]
tbl = (g.mean().round(2).astype(str) + ' ± ' + g.std().round(2).astype(str)).reindex([m for m in list(MODE_CFG) + list(SWEEP) if m in df['mode'].unique()])
tbl['n_seeds'] = df.groupby('mode').size()
print(tbl.to_string())
tbl.to_csv(WORK/'results_table_budget60.csv'); open(WORK/'results_table_budget60.tex', 'w').write(tbl.to_latex())

# reproducibility record
env = dict(torch=torch.__version__, ultralytics=ultralytics.__version__, gpu=torch.cuda.get_device_name(0),
           epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seeds=SEEDS, split_seed=SPLIT_SEED,
           assigner_defaults=OT._DEFAULT_CFG, modes=MODE_CFG, sweep=SWEEP)
json.dump(env, open(WORK/'experiment_config_budget60.json', 'w'), indent=1, default=str)
print('\nsaved: results_budget60.csv, results_table_budget60.csv/.tex, stability_results.csv, experiment_config_budget60.json')

                   AP          AP50 AP_vtiny(<8) AP_tiny(8-16) AP_small(16-32)  AP_rest(>32)  n_seeds
mode                                                                                                 
default  16.05 ± 0.19   27.12 ± 0.3  0.47 ± 0.12   2.58 ± 0.02     8.96 ± 0.03  25.42 ± 0.36        2
ot_nwd   15.12 ± 0.43  27.13 ± 0.63  0.38 ± 0.03    3.0 ± 0.11      9.5 ± 0.01   24.3 ± 0.61        2

saved: results_budget60.csv, results_table_budget60.csv/.tex, stability_results.csv, experiment_config_budget60.json
